# Loop Detection in Comparison Graph

Finds cycles of any length in the pairwise comparison graph using two strategies:

| Strategy | Best for | Complexity |
|---|---|---|
| **Exhaustive scan** | 2-cycles only | O(E) — no sampling, finds all |
| **DFS with depth limit** | Short cycles (3–6) | O(sample^L) — exact within sample |
| **Random walk** | Any length (7, 20, 100…) | O(steps) — probabilistic, finds representative cycles |

Each loop is printed with nodes ordered from **highest to lowest DB score**,
and the closing edge's full comparison record is shown.


In [ ]:
import time
import random
import pathlib
from tqdm import tqdm as tqdm_desc

from comfyui_image_scorer.domain.graph.chain_manager import ChainManager, tarjan_scc
from comfyui_image_scorer.infrastructure.persistence.comparisons_repository import (
    list_links,
    remove_comparison,
)
from comfyui_image_scorer.infrastructure.persistence.images_repository import list_nodes
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path
from comfyui_image_scorer.core.observability.logger import (
    get_logger,
    configure_package_logging,
)


from comfyui_image_scorer.infrastructure.persistence.images_repository import (
    find_node as get_image,
    update_image_rating_state,
)

configure_package_logging(30)
logger = get_logger(__name__)

# ── Configuration ─────────────────────────────────────────────────────────────

# -- 2-cycle exhaustive scan --
# Cap on how many 2-cycles to display (all are detected, this limits output).
MAX_2CYCLES: int = 100

# -- DFS exact search (3–6 cycles) --
# Maximum cycle length to search for with DFS. Each +1 multiplies cost by ~N.
# Practical range: 3–6. Above 6, use random walk instead.
MAX_EXACT_CYCLE_LEN: int = 1000

# Max cycles to collect per SCC from DFS search.
MAX_EXACT_CYCLES_PER_SCC: int = 0

# How many SCC members to use as DFS start nodes.
# Keep this small when MAX_EXACT_CYCLE_LEN >= 5 to avoid long runtimes.
DFS_SAMPLE_SIZE: int = 10000

# -- Random walk (any cycle length) --
# Only report cycles whose length falls in [min, max]. Set max=999 for no upper limit.
RW_MIN_CYCLE_LEN: int = 0
RW_MAX_CYCLE_LEN: int = 999

# Random walk attempts per SCC (each walk finds at most one cycle).
RW_ATTEMPTS_PER_SCC: int = 100

# Max steps per random walk before giving up on that walk.
RW_MAX_STEPS: int = 50000

# Max random-walk cycles to collect per SCC.
MAX_RW_CYCLES_PER_SCC: int = 100

# choose only the N max elements to remove from database, sorted by score difference (loser - winner)
MAX_FILTER_RANGE = 3


## 1. Load Data

In [ ]:
t0 = time.perf_counter()
images = list_nodes()
comparisons = list_links()
elapsed = time.perf_counter() - t0

db_score: dict[str, float] = {
    img["filename"]: float(img.get("score") or 0.0) for img in images
}
all_filenames = set(db_score.keys())

print(
    f"Loaded {len(comparisons)} comparisons across {len(all_filenames)} images in {elapsed:.2f}s"
)


## 2. Build Graph and Run Tarjan SCC

In [ ]:
cm = ChainManager()
cm._reset_adjacency()
cm._build_from_comparisons(comparisons)
cm._all_filenames = all_filenames

# directed edge -> comparison dict, keeping most recent per edge
edge_to_comp: dict[tuple[str, str], dict] = {}
for comp in comparisons:
    winner = comp["winner"]
    loser = comp["filename_b"] if winner == comp["filename_a"] else comp["filename_a"]
    key = (winner, loser)
    if key not in edge_to_comp or comp["id"] > edge_to_comp[key]["id"]:
        edge_to_comp[key] = comp

t0 = time.perf_counter()
scc_id_of, scc_members = tarjan_scc(cm._all_filenames, cm._worse_than)
elapsed = time.perf_counter() - t0
print(f"Tarjan SCC: {elapsed:.3f}s, {len(scc_members)} total SCCs")

cyclic_sccs = {sid: members for sid, members in scc_members.items() if len(members) > 1}
total_cyclic = sum(len(m) for m in cyclic_sccs.values())
sizes = sorted((len(m) for m in cyclic_sccs.values()), reverse=True)
print(f"Cyclic SCCs: {len(cyclic_sccs)}, total cyclic nodes: {total_cyclic}")
print(f"SCC sizes (top 20): {sizes[:20]}")


## 3. Cycle Search

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────


def rotate_to_highest_score(cycle_nodes: list[str]) -> list[str]:
    best_idx = max(
        range(len(cycle_nodes)), key=lambda i: db_score.get(cycle_nodes[i], 0.0)
    )
    return cycle_nodes[best_idx:] + cycle_nodes[:best_idx]


def fmt_comp(comp: dict | None) -> str:
    if comp is None:
        return "[closing comparison NOT found in DB]"
    return (
        f"id={comp['id']}  "
        f"filename_a='{comp['filename_a']}'  "
        f"filename_b='{comp['filename_b']}'  "
        f"winner='{comp['winner']}'  "
        f"timestamp='{comp['timestamp']}'"
    )


# collected across all search strategies for the image viewer
all_found_loops: list[tuple[str, list[str], list[dict | None]]] = []


def collect_and_print(label: str, cycle: list[str]) -> None:
    rotated = rotate_to_highest_score(cycle)
    loop_edges = [
        edge_to_comp.get((rotated[i], rotated[(i + 1) % len(rotated)]))
        for i in range(len(rotated))
    ]
    all_found_loops.append((label, rotated, loop_edges))
    scores = ", ".join(f"{n} ({db_score.get(n, 0):.2f})" for n in rotated)
    print(
        f"  [{label}] len={len(rotated)}"
    )  #: {" -> ".join(rotated)} -> {rotated[0]}')
    print(f"    scores : {scores[:20]}...")
    # print(f"    closing: {fmt_comp(loop_edges[-1])}")


print("Helpers ready.")


In [ ]:
# ── Strategy 2: DFS exact search for cycles of length 3 to MAX_EXACT_CYCLE_LEN ──
#
# For each sampled start node, runs a depth-limited DFS looking for any path
# that returns to the start. Exact within the sample, but cost is O(sample^L).
#
# Practical guidance:
#   L=3  → sample up to 300 nodes comfortably
#   L=4  → sample up to 100 nodes
#   L=5  → sample up to 30 nodes
#   L=6+ → use random walk instead

print(
    f"=== DFS exact cycles, length 3–{MAX_EXACT_CYCLE_LEN}, sample={DFS_SAMPLE_SIZE} ==="
)
cycles_scc = sorted(cyclic_sccs.items(), key=lambda x: -len(x[1]))
with tqdm_desc(desc="DFS SCCs", unit="SCC", total=len(cycles_scc)) as pbar:
    for scc_id, members in cycles_scc:
        node_set = set(members)
        sample = members[:DFS_SAMPLE_SIZE]
        succs = {
            u: [v for v in cm._worse_than.get(u, []) if v in node_set] for u in sample
        }
        succs_set = {u: set(vs) for u, vs in succs.items()}

        seen_dfs: set[tuple] = set()
        found = 0
        pbar.total = len(sample)

        for start in sample:
            if found >= MAX_EXACT_CYCLES_PER_SCC:
                break
            # DFS: stack entries are (current_node, path_so_far)
            stack = [(start, [start])]
            while stack and found < MAX_EXACT_CYCLES_PER_SCC:
                node, path = stack.pop()
                if len(path) > MAX_EXACT_CYCLE_LEN:
                    continue
                for nbr in succs.get(node, []):
                    if nbr == start and len(path) >= 3:  # 2-cycles handled separately
                        c = path[:]  # path is the cycle
                        min_idx = c.index(min(c))
                        canon = tuple(c[min_idx:] + c[:min_idx])
                        if canon not in seen_dfs:
                            seen_dfs.add(canon)
                            collect_and_print(f"SCC {scc_id} DFS #{found + 1}", c)
                            found += 1
                    elif nbr not in path and len(path) < MAX_EXACT_CYCLE_LEN:
                        stack.append((nbr, path + [nbr]))
            pbar.update(1)
print(f"\nDFS search complete.")


In [ ]:
# ── Strategy 3: Random walk — finds cycles of any length ─────────────────────
#
# Starts from a random SCC node, follows edges randomly, and stops when it
# revisits any node — extracting the cycle between the two visits.
# Length of cycles found depends on graph structure, not configuration.
# Use RW_MIN_CYCLE_LEN / RW_MAX_CYCLE_LEN to filter by cycle length.
#
# Increase RW_ATTEMPTS_PER_SCC to sample more of the cycle-length distribution.

print(f"=== Random walk — cycle lengths [{RW_MIN_CYCLE_LEN}, {RW_MAX_CYCLE_LEN}] ===")
print(f"    {RW_ATTEMPTS_PER_SCC} attempts per SCC, max {RW_MAX_STEPS} steps each\n")

rng = random.Random()
for scc_id, members in sorted(cyclic_sccs.items(), key=lambda x: -len(x[1])):
    node_set = set(members)
    # Restrict successors to the SCC so the walk stays inside
    succs = {
        u: [v for v in cm._worse_than.get(u, []) if v in node_set] for u in members
    }
    eligible = [u for u in members if succs.get(u)]  # nodes with at least one out-edge
    if not eligible:
        continue

    seen_rw: set[tuple] = set()
    found_rw = 0

    for _ in range(RW_ATTEMPTS_PER_SCC):
        if found_rw >= MAX_RW_CYCLES_PER_SCC:
            break

        start = rng.choice(eligible)
        path = [start]
        visited_idx: dict[str, int] = {start: 0}

        for _step in range(RW_MAX_STEPS):
            current = path[-1]
            neighbors = succs.get(current, [])
            if not neighbors:
                break
            nxt = rng.choice(neighbors)
            if nxt in visited_idx:
                # extract the cycle between the two visits to nxt
                cycle = path[visited_idx[nxt] :]
                clen = len(cycle)
                if RW_MIN_CYCLE_LEN <= clen <= RW_MAX_CYCLE_LEN:
                    min_idx = cycle.index(min(cycle))
                    canon = tuple(cycle[min_idx:] + cycle[:min_idx])
                    if canon not in seen_rw:
                        seen_rw.add(canon)
                        collect_and_print(
                            f"SCC {scc_id} RW #{found_rw + 1} (len={clen})", cycle
                        )
                        found_rw += 1
                break  # restart walk regardless of whether we kept this cycle
            path.append(nxt)
            visited_idx[nxt] = len(path) - 1

print(f"\nTotal loops collected (all strategies): {len(all_found_loops)}")


3.5 sort and filter by score difference

In [ ]:
filtered_comparisons = []

for loop in all_found_loops:
    loop_edges = loop[2]
    # filter by only lower scores beating higher scores
    loop_edges = [
        comp
        for comp in loop_edges
        if comp is not None
        and db_score[comp["winner"]]
        < db_score[
            (
                comp["filename_a"]
                if comp["winner"] == comp["filename_b"]
                else comp["filename_b"]
            )
        ]
    ]
    # sort edges by score difference
    loop_edges.sort(
        key=lambda comp: (
            abs(db_score[comp["filename_a"]] - db_score[comp["filename_b"]])
            if comp is not None
            else -1
        ),
        reverse=True,
    )
    title = loop[0]
    loop_names = set()
    for comp in loop_edges[:MAX_FILTER_RANGE]:
        loop_names.add(comp["filename_a"])
        loop_names.add(comp["filename_b"])

    filtered_comparisons.append(
        (title, list(loop_names), loop_edges[:MAX_FILTER_RANGE])
    )
print(len(filtered_comparisons))


## 4. Visual Comparison Viewer

Shows each edge of every collected loop as a side-by-side image pair.  
**Left (green border)** = winner.  
**Right (red border)** = loser.  
The **closing edge** (the one that creates the contradiction) has a **gold border**.


In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

_SIDE_STYLES = (("WINNER", "tab:green"), ("LOSER", "tab:red"))


def show_loop_images(
    label: str, loop_nodes: list[str], loop_edges: list[dict | None]
) -> None:
    n_edges = len(loop_edges)
    fig, axes = plt.subplots(n_edges, 2, figsize=(10, 4 * n_edges), squeeze=False)
    fig.suptitle(label, fontsize=13, fontweight="bold")

    for row_i in range(n_edges):
        comp = loop_edges[row_i]
        is_closing = row_i == n_edges - 1
        # src = loop_edges[row_i]
        # dst = loop_edges[(row_i + 1) % n_edges]

        # if comp is not None:
        winner_fn = comp["winner"]
        loser_fn = (
            comp["filename_b"]
            if comp["winner"] == comp["filename_a"]
            else comp["filename_a"]
        )
        edge_id, edge_ts = comp["id"], comp["timestamp"]
        # else:
        #     winner_fn, loser_fn = src, dst
        #     edge_id, edge_ts = None, "N/A"

        closing_tag = "  [CLOSING EDGE]" if is_closing else ""
        row_title = f"Edge {row_i + 1}{closing_tag}  |  id={edge_id}  ts={edge_ts}"

        for col, filename in enumerate((winner_fn, loser_fn)):
            side_label, side_color = _SIDE_STYLES[col]
            ax = axes[row_i][col]
            path = find_image_path(filename)
            if path is None:
                ax.text(
                    0.5,
                    0.5,
                    f"not found:\n{filename}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    wrap=True,
                )
            else:
                ax.imshow(mpimg.imread(path))
            score_val = db_score.get(filename, 0.0)
            title = f"{side_label} | score={score_val:.3f}\n{filename}"
            if col == 0:
                title = f"{row_title}\n{title}"
            ax.set_title(title, fontsize=7.5, color=side_color)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor("gold" if is_closing else side_color)
                spine.set_linewidth(4 if is_closing else 3)

    fig.subplots_adjust(hspace=0.5, wspace=0.06)
    plt.show()


# for loop_label, loop_nodes, loop_edges in all_found_loops[:3]:
#     show_loop_images(loop_label, loop_nodes, loop_edges)

for loop_label, loop_nodes, loop_edges in filtered_comparisons[:3]:
    show_loop_images(loop_label, loop_nodes, loop_edges)


In [ ]:
def _persist_image_state(filename: str, data: dict) -> bool:
    return update_image_rating_state(
        filename=filename,
        score=float(data["score"]),
        rating_mu=float(data["rating_mu"]),
        rating_sigma=float(data["rating_sigma"]),
        comparison_count=int(data["comparison_count"]),
        touch_timestamp=False,
    )


In [ ]:
confirmed_pairs = []

for comparison in filtered_comparisons:
    loop_edges = comparison[2]
    for comp in loop_edges:
        confirmed_pairs.append(
            (
                comp["winner"],
                (
                    comp["filename_b"]
                    if comp["winner"] == comp["filename_a"]
                    else comp["filename_a"]
                ),
            )
        )
print(f"{len(confirmed_pairs)} confirmed loop pairs to remove from database")

confirmed_pairs = list(set(confirmed_pairs))  # deduplicate
print(f"{len(confirmed_pairs)} unique confirmed loop pairs to remove from database")

dry_run = False  # Set to False to actually write to DB
if not dry_run:
    removed_count = 0
    for pair in tqdm_desc(
        confirmed_pairs, desc="Removing from DB", total=len(confirmed_pairs), delay=3
    ):
        comparison_id = remove_comparison(
            filename_a=pair[0],
            filename_b=pair[1],
        )
        winner_data = get_image(pair[0])
        loser_data = get_image(pair[1])
        winner_data["comparison_count"] -= 1
        loser_data["comparison_count"] -= 1

        _persist_image_state(pair[0], winner_data)
        _persist_image_state(pair[1], loser_data)

        removed_count += 1

    print(f"Removed {removed_count} loop pairs to database")
print("Experiment complete - no files were modified")
